In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ndcg_score

In [2]:
# TCVファイルの読み込み
train_A = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")
train_B = pd.read_csv('../000.data/train/train_B.tsv', sep="\t")
train_C = pd.read_csv('../000.data/train/train_C.tsv', sep="\t")
train_D = pd.read_csv('../000.data/train/train_D.tsv', sep="\t")
test_data = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# trainを一つに
train_data = pd.concat([train_A, train_B, train_C, train_D], ignore_index=True)

In [4]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:  # 広告経由の購入
        return 3
    elif row["event_type"] == 2:  # 広告クリック
        return 2
    elif row["event_type"] == 1:  # 詳細ページ閲覧
        return 1
    else:  # カート追加
        return 0

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)

In [5]:
# タイムスタンプを数値型に変換
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])
train_data['time_stamp'] = train_data['time_stamp'].astype(np.int64) // 10**9  # UNIX時間に変換

In [6]:
# ユーザIDと商品IDのエンコード
user_enc = LabelEncoder()
product_enc = LabelEncoder()

train_data['user_id'] = user_enc.fit_transform(train_data['user_id'])
train_data['product_id'] = product_enc.fit_transform(train_data['product_id'])

test_data[('user_id')] = user_enc.transform(test_data['user_id'])

In [7]:
# 組み合わせ特徴量の作成
interaction_features = train_data.groupby(['user_id', 'product_id']).agg(
    click_count=('relevance', lambda x: (x == 2).sum()),
    purchase_count=('relevance', lambda x: (x == 3).sum()),
    last_interaction_time=('time_stamp', 'max')
).reset_index()

In [8]:
# 組み合わせ特徴量をtrain_dataにマージ
train_data = pd.merge(train_data, interaction_features, on=['user_id', 'product_id'], how='left')


In [9]:
# XGBoost 用データ作成
train_X = train_data[['user_id', 'product_id', 'ad', 'time_stamp', 'click_count', 'purchase_count']]
train_y = train_data['relevance']

In [10]:
# XGBoostのモデルとパラメータグリッド
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse')

param_grid = {
    'learning_rate': [0.025, 0.03],
    'max_depth': [5, 6],
    'n_estimators': [75, 100],
    'subsample': [0.7],
    'colsample_bytree': [0.8, 0.9],
    'min_child_weight': [4, 5],
    'gamma': [0, 0.3],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1.0, 1.5],
    'tree_method': ['hist'],
    'grow_policy': ['lossguide']
}

In [11]:
# グリッドサーチの設定
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=1, verbose=2)

In [12]:
# グリッドサーチ実行
grid_search.fit(train_X, train_y)

Fitting 3 folds for each of 256 candidates, totalling 768 fits
[CV] END colsample_bytree=0.8, gamma=0, grow_policy=lossguide, learning_rate=0.025, max_depth=5, min_child_weight=4, n_estimators=75, reg_alpha=0, reg_lambda=1.0, subsample=0.7, tree_method=hist; total time=   3.6s
[CV] END colsample_bytree=0.8, gamma=0, grow_policy=lossguide, learning_rate=0.025, max_depth=5, min_child_weight=4, n_estimators=75, reg_alpha=0, reg_lambda=1.0, subsample=0.7, tree_method=hist; total time=   3.3s
[CV] END colsample_bytree=0.8, gamma=0, grow_policy=lossguide, learning_rate=0.025, max_depth=5, min_child_weight=4, n_estimators=75, reg_alpha=0, reg_lambda=1.0, subsample=0.7, tree_method=hist; total time=   4.1s
[CV] END colsample_bytree=0.8, gamma=0, grow_policy=lossguide, learning_rate=0.025, max_depth=5, min_child_weight=4, n_estimators=75, reg_alpha=0, reg_lambda=1.5, subsample=0.7, tree_method=hist; total time=   4.7s
[CV] END colsample_bytree=0.8, gamma=0, grow_policy=lossguide, learning_rate=

GridSearchCV(cv=3,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False,
                                    eval_metric='rmse', feature_types=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,
                                    learning_rate=None...
                                    n_jobs=None, num_parallel_tree=None,
                                    random_state=None, ...),
             n_jobs=1,
             param_grid={'colsample_bytree': [0.8, 0.9], 'gamma': [0, 0.3],
                         'grow_policy': ['lossguide'],
                         'learning_rate': [0.025, 0.03], 'max_depth': [5, 6],
                         'min_child_weight': [4, 5], 'n_estimators': [75, 100],
                         'reg_alpha': [0, 0.1], 'reg_lambda': [1.0, 1.5],
                         'subsample': [0.7], 'tree_method': ['hist']},
             scoring='neg_mean_squared_error', verbose=2)

In [13]:
# 最適パラメータと最良スコアの表示
print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

Best parameters: {'colsample_bytree': 0.8, 'gamma': 0, 'grow_policy': 'lossguide', 'learning_rate': 0.025, 'max_depth': 6, 'min_child_weight': 4, 'n_estimators': 75, 'reg_alpha': 0, 'reg_lambda': 1.0, 'subsample': 0.7, 'tree_method': 'hist'}
Best score: -0.2965262134869893


In [14]:
# 最適なモデルで再学習
best_model = grid_search.best_estimator_

In [15]:
# 推薦関数
def recommend_products(user_id, top_k=22):
    user_data = train_data[train_data['user_id'] == user_id][['user_id', 'product_id', 'ad', 'time_stamp', 'click_count', 'purchase_count']]
    if user_data.empty:
        print(f"Warning: User {user_id} has no data.")
        return pd.DataFrame(columns=['product_id', 'rank'])
    predictions = best_model.predict(user_data)
    user_data['score'] = predictions
    recommendations = user_data.sort_values(by='score', ascending=False).head(top_k)
    recommendations['rank'] = range(1, len(recommendations) + 1)
    recommendations['user_id'] = user_enc.inverse_transform(recommendations['user_id'])
    recommendations['product_id'] = product_enc.inverse_transform(recommendations['product_id'])
    return recommendations[['user_id', 'product_id', 'rank']]

In [16]:
# nGCDの計算
def evaluate_ndcg():
    y_true = []
    y_score = []
    
    for user_id in test_data['user_id'].unique():
        if user_id in train_data['user_id'].values:
            actual = train_data[train_data['user_id'] == user_id].sort_values(by='relevance', ascending=False)['relevance'].values
            
            # スコアの予測
            user_data = train_data[train_data['user_id'] == user_id][['user_id', 'product_id', 'ad', 'time_stamp', 'click_count', 'purchase_count']]
            if user_data.empty:
                print(f"Warning: User {user_id} has no data.")
                continue
            
            predictions = best_model.predict(user_data)
            
            # 実際の関連度と予測スコアをリストに追加
            y_true.append(actual)
            y_score.append(predictions)
    
    # nDCG計算
    return np.mean([ndcg_score([t], [s]) for t, s in zip(y_true, y_score)]) if y_true else 0.0

In [17]:
# 予測結果保存
def save_predictions():
    results = []
    for user_id in test_data['user_id'].unique():
        if user_id in train_data['user_id'].values:
            recommendations = recommend_products(user_id)
            for _, row in recommendations.iterrows():
                results.append([row['user_id'], row['product_id'], row['rank']])
    if not results:
        print("Warning: No predictions generated.")
    pd.DataFrame(results, columns=['user_id', 'product_id', 'rank']).to_csv('predictions.tsv', sep='\t', index=False, header=False)

In [18]:
# 推薦結果表示
print("nDCG Score:", evaluate_ndcg())
save_predictions()

nDCG Score: 0.8584346836346537
